# Source Isochrone Systematics

This notebook compares the default PARSEC source-forward prior with MIST solar-scaled and alpha-enhanced alternatives. It uses PRIME photometry so the source selection can be written as an apparent `Imag` range, and it inspects the resulting source physical properties. See `docs/isochrone_systematics.md` for the design notes and caveats.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "CMakeLists.txt").exists() and (path / "build").exists():
            return path
    return start

repo_root = find_repo_root(Path.cwd().resolve())
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens

pd.set_option("display.max_rows", 80)
repo_root

## Configuration

The selection uses an apparent PRIME `Imag` range with the same automatic extinction model used by the source-forward examples. Adjust `N_SIMU` upward for publication-quality figures.

In [ ]:
N_SIMU = 100_000
SEED = 2026
PHOTOMETRY = "prime"
I_MIN = 12.0
I_MAX = 21.0
CORNER_COLUMNS = ["M_L", "D_L", "D_S", "mu_rel_N", "mu_rel_E"]
SOURCE_COLUMNS = ["iS", "D_S", "logage_S", "MH_S", "M_S_ini", "M_S", "R_S", "teff_S", "logg_S", "theta_S", "M_Imag_S", "M_Vmag_S", "VI_S"]
SOURCE_PROPERTY_COLUMNS = ["M_S_ini", "M_S", "R_S", "teff_S", "logg_S", "theta_S", "M_Imag_S", "M_Vmag_S", "VI_S"]
SOURCE_CORNER_COLUMNS = ["D_S", "M_S_ini", "R_S", "teff_S", "M_Imag_S"]


def as_dataframe(result):
    return pd.DataFrame(result.to_numpy(), columns=result.columns)


def weighted_quantile(values, weights, quantiles=(0.16, 0.50, 0.84)):
    values = np.asarray(values)
    weights = np.asarray(weights)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if len(values) == 0:
        return [np.nan for _ in quantiles]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(quantiles, cdf, values)


def base_config():
    cfg = genulens.Config(l=1.0, b=-3.9, n_simu=N_SIMU, seed=SEED)
    cfg.source.mode = "isochrone"
    cfg.source.photometry = PHOTOMETRY
    cfg.source.band = "Imag"
    cfg.source.min_magnitude = I_MIN
    cfg.source.max_magnitude = I_MAX
    cfg.source.min_initial_mass_msun = 0.1
    cfg.source.max_initial_mass_msun = 2.0
    cfg.source.extinction_mode = "genstars"
    cfg.source.extinction_law = 1
    cfg.source.ejk_rc = 1.0
    cfg.source.dm_rc = 14.5
    cfg.observation.IL_err = 0.0
    return cfg


def alpha_config(family, fraction):
    cfg = base_config()
    cfg.source.isochrone_model = "alpha_mixture" if fraction > 0 else "parsec_solar_scaled"
    cfg.source.secondary_isochrone_family = family
    cfg.source.secondary_isochrone_abundance = "alpha_enhanced"
    cfg.source.secondary_isochrone_alpha_fe = 0.4
    cfg.source.alpha_enhanced_fraction = 0.0
    cfg.source.alpha_enhanced_components = [8, 9]
    cfg.source.alpha_enhanced_component_fractions = [fraction, fraction]
    return cfg

## Run Simulations

In [ ]:
runs = {
    "parsec_solar": base_config(),
    "mist_alpha_0p5": alpha_config("mist", 0.5),
}

dfs = {}
for label, cfg in runs.items():
    result = genulens.simulate(cfg)
    dfs[label] = as_dataframe(result)
    if {"M_Vmag_S", "M_Imag_S"}.issubset(dfs[label].columns):
        dfs[label]["VI_S"] = dfs[label]["M_Vmag_S"] - dfs[label]["M_Imag_S"]
    print(label, dfs[label].shape)

assert all(len(df) == N_SIMU for df in dfs.values())

## Weighted Source-Property Summary

In [ ]:
summary_rows = []
for label, df in dfs.items():
    weights = df["wtj"].to_numpy()
    for column in SOURCE_COLUMNS:
        if column not in df:
            continue
        q16, q50, q84 = weighted_quantile(df[column], weights)
        summary_rows.append({"run": label, "column": column, "q16": q16, "q50": q50, "q84": q84})

summary = pd.DataFrame(summary_rows)
summary

## Source-Property Histograms

In [ ]:
plot_columns = ["teff_S", "R_S", "logg_S", "theta_S", "M_Imag_S", "VI_S"]
fig, axes = plt.subplots(len(plot_columns), 1, figsize=(8, 12), constrained_layout=True)
for ax, column in zip(axes, plot_columns):
    values = [df[column].replace([np.inf, -np.inf], np.nan).dropna().to_numpy() for df in dfs.values()]
    finite = np.concatenate([v for v in values if len(v)])
    bins = np.linspace(np.nanpercentile(finite, 1), np.nanpercentile(finite, 99), 45)
    for label, df in dfs.items():
        mask = np.isfinite(df[column]) & np.isfinite(df["wtj"]) & (df["wtj"] > 0)
        ax.hist(df.loc[mask, column], bins=bins, weights=df.loc[mask, "wtj"], histtype="step", density=True, label=label)
    ax.set_xlabel(column)
    ax.set_ylabel("weighted density")
    ax.legend(fontsize=8)
plt.show()

## HR Diagram

The scatter plot below shows the source stars in an HR-diagram-like plane. The x-axis is `teff_S` and the y-axis is the absolute `Imag` magnitude from the active PRIME isochrone table.

In [ ]:
fig, axes = plt.subplots(
    1, len(dfs),
    figsize=(5 * len(dfs), 4),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

if len(dfs) == 1:
    axes = [axes]

sc = None

for ax, (label, df) in zip(axes, dfs.items()):
    mask = (
        np.isfinite(df["teff_S"]) &
        np.isfinite(df["M_Imag_S"]) &
        np.isfinite(df["wtj"]) &
        (df["wtj"] > 0)
    )

    plot_df = df.loc[mask, ["teff_S", "M_Imag_S", "wtj", "iS"]].copy()

    if len(plot_df) > 4000:
        plot_df = plot_df.sample(4000, random_state=SEED)

    color = np.log10(plot_df["wtj"].to_numpy())

    sc = ax.scatter(
        plot_df["teff_S"],
        plot_df["M_Imag_S"],
        c=color,
        s=8,
        alpha=0.45,
        cmap="viridis",
        linewidths=0
    )

    ax.set_title(label)
    ax.set_xlabel("teff_S [K]")
    ax.set_ylabel("M_Imag_S")

# sharex/sharey なので代表軸だけ一度反転すれば十分
axes[0].invert_xaxis()
axes[0].invert_yaxis()

fig.colorbar(sc, ax=axes, label="log10(wtj)")
plt.show()

## Source Component Mix

The selected source prior can change the component mix. The table and bar plot below show the weighted fraction of sampled source components after the apparent `Imag` selection.

In [ ]:
component_rows = []
for label, df in dfs.items():
    weights = df["wtj"].to_numpy()
    total = np.sum(weights[np.isfinite(weights) & (weights > 0)])
    for component in sorted(df["iS"].dropna().unique()):
        mask = (df["iS"] == component) & np.isfinite(df["wtj"]) & (df["wtj"] > 0)
        component_rows.append({"run": label, "iS": int(component), "weighted_fraction": df.loc[mask, "wtj"].sum() / total})

component_mix = pd.DataFrame(component_rows)
display(component_mix.pivot(index="iS", columns="run", values="weighted_fraction").fillna(0.0))

ax = component_mix.pivot(index="iS", columns="run", values="weighted_fraction").fillna(0.0).plot(kind="bar", figsize=(8, 3))
ax.set_xlabel("source component iS")
ax.set_ylabel("weighted fraction")
plt.tight_layout()
plt.show()


## Source-Property Corner Plot

This corner plot focuses on the source-side quantities that are useful for forward-prior and limb-darkening work.

In [ ]:
def finite_weighted_values(df, columns):
    mask = np.isfinite(df["wtj"]) & (df["wtj"] > 0)
    for column in columns:
        mask &= np.isfinite(df[column])
    return df.loc[mask, columns].to_numpy(), df.loc[mask, "wtj"].to_numpy()

try:
    import corner

    for label, df in dfs.items():
        values, weights = finite_weighted_values(df, SOURCE_CORNER_COLUMNS)
        fig = corner.corner(values, labels=SOURCE_CORNER_COLUMNS, weights=weights, quantiles=[0.16, 0.50, 0.84], show_titles=True, title_fmt=".3g")
        fig.suptitle(f"{label}: source properties, {I_MIN:g} <= I_S <= {I_MAX:g}", y=1.02)
        plt.show()
except ImportError:
    for label, df in dfs.items():
        axes = pd.plotting.scatter_matrix(df[SOURCE_CORNER_COLUMNS], figsize=(9, 9), diagonal="hist", alpha=0.08)
        axes[0, 0].figure.suptitle(f"{label}: source properties", y=1.02)
        plt.show()


## Event-Prior Corner Plot

The event-level prior may change when the selected-source probability changes with the alternate isochrone table. The plot below uses the same columns as the source-forward smoke comparison.

In [ ]:
def filtered_corner_data(df):
    mu_abs = np.sqrt(df["mu_rel_N"]**2 + df["mu_rel_E"]**2)
    out = df[(mu_abs <= 20) & (df["M_L"] <= 1.5)].copy()
    return out[CORNER_COLUMNS].to_numpy(), out["wtj"].to_numpy()

try:
    import corner

    for label, df in dfs.items():
        values, weights = filtered_corner_data(df)
        fig = corner.corner(values, labels=CORNER_COLUMNS, weights=weights, quantiles=[0.16, 0.50, 0.84], show_titles=True, title_fmt=".3g")
        fig.suptitle(label, y=1.02)
        plt.show()
except ImportError:
    fig, axes = plt.subplots(1, len(CORNER_COLUMNS), figsize=(14, 3), constrained_layout=True)
    for ax, column in zip(axes, CORNER_COLUMNS):
        for label, df in dfs.items():
            mask = np.isfinite(df[column]) & np.isfinite(df["wtj"]) & (df["wtj"] > 0)
            ax.hist(df.loc[mask, column], bins=40, weights=df.loc[mask, "wtj"], histtype="step", density=True, label=label)
        ax.set_xlabel(column)
    axes[0].legend(fontsize=8)
    plt.show()

## Caveats

The MIST tables used here cover source component indices 0-10 and are useful for testing sensitivity to alpha-enhanced stellar populations. Differences between PARSEC and MIST include more than `[alpha/Fe]`: stellar physics, bolometric corrections, passbands, and zero-points also differ.